# BytePlus Registration - Run from Colab (US IP)
Just click **Runtime > Run All** — takes ~3 minutes.

In [ ]:
import requests, re, time, random, string, json

ARK_CN = 'https://ark.cn-beijing.volces.com/api/v3'
MODEL = 'doubao-seedance-2-5-260628'
H = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# Check IP
ip_info = requests.get('https://ipinfo.io/json', timeout=10).json()
print(f'IP: {ip_info.get("ip")} | {ip_info.get("org")} | {ip_info.get("country")}')

# Check __spti bucket
S = requests.Session()
S.headers.update({**H, 'Accept': 'text/html'})
r0 = S.get('https://console.byteplus.com/auth/signup', timeout=12)
spti = S.cookies.get('__spti', '')
print(f'__spti: {spti}')
print(f'Risk flagged: {"bapxu9rXuEQMtPL" in spti}')
print('If flagged=False -> this will work!')

In [ ]:
results = []

for attempt in range(5):
    print(f'\n--- Attempt {attempt+1} ---')
    
    USERNAME = 'bpdev' + ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    PASSWORD = 'BpP1us!2026#A'
    
    # Get guerrillamail
    gm = requests.get('https://api.guerrillamail.com/ajax.php?f=get_email_address',
                      headers={'User-Agent': 'Mozilla/5.0'}, timeout=10).json()
    _user = gm.get('email_addr', '').split('@')[0]
    sid = gm.get('sid_token', '')
    requests.get('https://api.guerrillamail.com/ajax.php',
                 params={'f': 'set_email_user', 'email_user': _user, 'lang': 'en',
                         'sid_token': sid, 'in': 'guerrillamail.com'},
                 headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
    email = f'{_user}@guerrillamail.com'
    print(f'Email: {email}, Username: {USERNAME}')
    
    S = requests.Session()
    S.headers.update({**H, 'Accept': 'text/html'})
    S.get('https://console.byteplus.com/auth/signup', timeout=12)
    spti = S.cookies.get('__spti', '')
    print(f'  spti: {spti} flagged={"bapxu9rXuEQMtPL" in spti}')
    
    S.headers.update({
        'Accept': 'application/json',
        'Origin': 'https://console.byteplus.com',
        'Referer': 'https://console.byteplus.com/auth/signup',
        'X-Csrf-Token': '8b13c7ddf17a645320fff792b453c01c',
    })
    
    # Send OTP
    r_otp = S.post('https://console.byteplus.com/api/passport/login/sendSignupEmailforBP',
                   json={'Email': email, 'AccountType': 'byteplus'}, timeout=15)
    otp_ec = ''
    try: otp_ec = r_otp.json().get('ResponseMetadata', {}).get('Error', {}).get('Code', '')
    except: pass
    print(f'  OTP: {r_otp.status_code} ec={otp_ec!r}')
    if r_otp.status_code != 200 or otp_ec:
        continue
    
    # Wait OTP
    otp = None
    for i in range(40):
        time.sleep(3)
        gm2 = requests.get(f'https://api.guerrillamail.com/ajax.php?f=check_email&seq=0&sid_token={sid}',
                           headers={'User-Agent': 'Mozilla/5.0'}, timeout=15).json()
        for mail in gm2.get('list', []):
            if any(x in mail.get('mail_subject', '').lower() for x in ['byteplus', 'verify', 'code']):
                mid = mail.get('mail_id')
                br = requests.get(f'https://api.guerrillamail.com/ajax.php?f=fetch_email&email_id={mid}&sid_token={sid}',
                                  headers={'User-Agent': 'Mozilla/5.0'}, timeout=15).json()
                body = br.get('mail_body', '') + br.get('mail_body_html', '')
                codes = re.findall(r'\\b(\\d{6})\\b', re.sub(r'<[^>]+>', ' ', body))
                if codes:
                    otp = codes[0]
                    break
        if otp:
            print(f'  OTP: {otp}')
            break
        if i < 5: print(f'  checking {(i+1)*3}s...')
    
    if not otp:
        print('  No OTP')
        continue
    
    # Register
    r_reg = S.post('https://console.byteplus.com/api/passport/login/signup',
                   json={'Email': email, 'AccountName': USERNAME, 'Password': PASSWORD,
                         'VerificationCode': otp, 'AccountType': 'byteplus',
                         'AcceptAgreements': ['byteplus_terms', 'byteplus_privacy']},
                   timeout=15)
    reg_ec = ''
    try: reg_ec = r_reg.json().get('ResponseMetadata', {}).get('Error', {}).get('Code', '')
    except: pass
    print(f'  Register: {r_reg.status_code} ec={reg_ec!r}')
    print(f'  Body: {r_reg.text[:300]}')
    
    if r_reg.status_code == 200 and not reg_ec:
        print(f'\n✅✅✅ SUCCESS! email={email} user={USERNAME} pass={PASSWORD}')
        results.append({'email': email, 'username': USERNAME, 'password': PASSWORD})
        break
    elif 'Risk' in reg_ec:
        print(f'  BLOCKED from this IP - try different Colab region/runtime')
        break
    
    time.sleep(5)

print(f'\nFinal results: {results}')

In [ ]:
# If registered, navigate to ARK and get API key
if results:
    creds = results[0]
    print(f'Registered account: {creds}')
    print(f'\nNow go to:')
    print(f'https://console.byteplus.com/ark/region:ark+cn-beijing/modelSquare/detail/model/doubao-seedance-2-5-260628')
    print(f'Login with email={creds["email"]} pass={creds["password"]}')
    print(f'Click "Open" on the model -> Create API Key -> copy UUID key')
else:
    print('No successful registrations')
    print('Check if Colab IP is flagged (spti shows bapxu9rXuEQMtPL)')